# Track A — Bước A8 (Đợt 3, Ưu tiên #1): Cộng hưởng tối thượng `A7a` (`DINOv3 Auto-ROI Fusion`) + `A6` (`Imbalanced-DYDM`) trên `GC10-DET`

- Nhánh: `duytung/13-gc10-dinov3-autoroi-dydm`
- Khởi tạo `Imbalanced-DYDM` trực tiếp từ cụm `N` vô địch của A7a (`bal = 0.5372`, `All = 76.88%`) trên tập `keep` (không bị lọc rớt lớp hiếm), tái sử dụng hoàn toàn đặc trưng & snapshot A7a trên Drive (~35 phút trên L4).
- Xuất `gc10_a8_autoroi_dydm_summary.txt` / `.json`.

In [ ]:
# 1. Kết nối Google Drive (bỏ qua nếu runtime hiện tại đã mount sẵn)
import os
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Google Drive đã được mount sẵn!')

In [ ]:
# 2. Cập nhật nhánh duytung/13-gc10-dinov3-autoroi-dydm
import os
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
REPO_DIR = '/content/VB-CGCD-defect-testing-only'
BRANCH = 'duytung/13-gc10-dinov3-autoroi-dydm'
os.makedirs(WORK_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://github.com/DuytungDeveloper/VB-CGCD-defect-testing-only.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin {BRANCH}
!git checkout -B {BRANCH} origin/{BRANCH}
!git status -sb

In [ ]:
# 3. Cài gói thiếu (bỏ qua nếu runtime hiện tại đã cài)
!pip uninstall -y -q torchao
!pip install -q numpyro continuum peft datasets transformers scipy scikit-learn psutil
!nvidia-smi

In [ ]:
# 4. Nạp HF_TOKEN và chạy nền Bước A8 (A7a + Imbalanced-DYDM)
import os, subprocess, time
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
LOG_FILE = f'{WORK_DIR}/gc10_a8_autoroi_dydm.log'
PID_FILE = f'{WORK_DIR}/gc10_a8_autoroi_dydm.pid'

hf_token = os.environ.get('HF_TOKEN', '')
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN') or ''
    except Exception:
        hf_token = ''
# hf_token = 'hf_...'

env = os.environ.copy()
if hf_token:
    env['HF_TOKEN'] = hf_token
with open(LOG_FILE, 'w') as f:
    p = subprocess.Popen(['bash', 'colab/run_gc10_a8_bg.sh', 'auto'], stdout=f, stderr=subprocess.STDOUT, env=env)
with open(PID_FILE, 'w') as f:
    f.write(str(p.pid))
print(f'Đã khởi chạy nền Bước A8 (A7a + Imbalanced-DYDM) với PID={p.pid}')
print(f'Log: {LOG_FILE}')
time.sleep(3)
!tail -n 20 {LOG_FILE}

In [ ]:
# 5. Kiểm tra tiến độ & xem kết quả + thông số phần cứng peak
import os
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
LOG_FILE = f'{WORK_DIR}/gc10_a8_autoroi_dydm.log'
PID_FILE = f'{WORK_DIR}/gc10_a8_autoroi_dydm.pid'
SUMMARY_OUT = f'{WORK_DIR}/gc10_a8_autoroi_dydm_summary.txt'

pid = open(PID_FILE).read().strip() if os.path.exists(PID_FILE) else ''
alive = False
if pid and os.path.exists(f'/proc/{pid}/status'):
    st = open(f'/proc/{pid}/status').read()
    alive = ('State:\tZ' not in st and 'State: Z' not in st)
print(f'Trạng thái PID={pid}: {"ĐANG CHẠY" if alive else "ĐÃ DỪNG / XONG"}')
print('\n--- 40 dòng log gần nhất ---')
if os.path.exists(LOG_FILE):
    lines = open(LOG_FILE).read().splitlines()
    print('\n'.join(lines[-40:]))
if os.path.exists(SUMMARY_OUT):
    print('\n--- BẢNG KẾT QUẢ & THÔNG SỐ PHẦN CỨNG PEAK BƯỚC A8 (GC10 A7a + Imbalanced-DYDM) ---')
    print(open(SUMMARY_OUT).read())